# Counting Arnold's Meanders in OCaml

### An OCaml companion to *Counting Arnold's Meanders with Verified Algorithms*

In 1988 Vladimir Arnold asked: in how many ways can a river cross a straight road $n$ times? The
river comes in from the south, crosses the road $n$ times without ever crossing itself, and
leaves to the east. The answers start $1, 1, 1, 2, 3, 8, 14, 42, 81, 262, \ldots$ (OEIS
A005316), and no formula is known.

The paper ([First Pair](https://firstpair.org/books/arnold-meanders/)) defines these numbers in
the Lean proof assistant and proves two fast algorithms correct. This notebook teaches OCaml by
building the same algorithms: the definition by brute force, the search with its parity rule,
and the transfer matrix, first on one core and then on all of them with OCaml 5's domains. The
last section explains how the repository's OxCaml program goes further, with OxCaml's
data-race-free parallelism and unboxed values. No OCaml is assumed.

**How the notebook teaches.** Every OCaml feature is explained once, in a text cell, before the
first code cell that uses it, and then used freely. Each function is defined once. The index at
the end lists every feature with the section that introduces it.

**OCaml is not Lean.** Lean proves the fast algorithms equal to the definition for every $n$.
Here we test them, against each other and against the values certified in Lean.

**Running it.** The notebook runs in the OCaml Jupyter kernel
([ocaml-jupyter](https://github.com/akabe/ocaml-jupyter)) with OCaml 5 and the `unix` library.
Run the cells in order from the top.

## Contents

1. OCaml as a calculator
2. Crossing chords
3. A river is a permutation
4. Checking what we cannot prove
5. Closed meanders
6. A search with a parity rule
7. The transfer matrix
8. All cores at once
9. OxCaml
10. Index of OCaml features

## 1. OCaml as a calculator

A code cell holds OCaml *phrases*. For each one, the toplevel prints its type and its value:
`- : int = 14` says the value is the integer 14. Comments are written `(* … *)`.

**Numbers.** `int` is a 63-bit integer: `max_int` is $2^{62} - 1$, and arithmetic past it wraps
round silently. `/` divides and rounds toward zero, and `mod` is the remainder. Floating-point
numbers have their own type, `float`, and their own operators `+.`, `*.`, `/.`; OCaml never
mixes the two silently.

In [1]:
2 + 3 * 4

- : int = 14


In [2]:
max_int

- : int = 4611686018427387903


In [3]:
(17 / 5, 17 mod 5, 17. /. 5.)

- : int * int * float = (3, 2, 3.4)


The last cell is a *tuple*. Its type, `int * int * float`, lists the types of its parts.

**`let` and printing.** `let name = expr` gives a value a name for the rest of the notebook.
`let x = a in e` names `a` only inside the expression `e`. `Printf.printf` prints with a format
string: `%d` is an integer, `%s` a string, `%.1f` a float with one decimal, `\n` a new line, and
`%!` flushes the output so that it appears at once. Strings are joined with `^`.

In [4]:
let q, r = 17 / 5, 17 mod 5
let () = Printf.printf "17 = 5 * %d + %d %s\n%!" q r ("(" ^ "checked" ^ ")")

val q : int = 3
val r : int = 2


17 = 5 * 3 + 2 (checked)


The first line also *destructures* a tuple: the pattern `q, r` names its two parts. The second
binds the pattern `()`, the only value of type `unit`. That is the usual way to run something
for its effect, here printing.

**Truth values.** `bool` has the values `true` and `false`. `=` and `<>` test equality and
inequality of any two values of the same type, comparing their structure. `&&`, `||` and `not`
combine truth values.

In [5]:
(3 < 5, 3 = 5, 3 <> 5, not (3 < 5) || true)

- : bool * bool * bool * bool = (true, false, true, true)


## 2. Crossing chords

Draw the road as a horizontal line and number its bridges $0, 1, \ldots, n-1$ from west to
east. Between consecutive crossings the river runs along an *arc*, below the road or above it,
and the arcs alternate sides. Two arcs on the same side are like two chords of a disk: they
cross exactly when their endpoints alternate. So a river is a meander exactly when no two arcs
on the same side alternate.

**Functions.** `let f a b = body` defines a function of two arguments, and `f 1 2` applies it.
`let a = … and b = … in` makes two definitions at once. Comparing truth values with `<>` is
*exclusive or*.

**Type inference and type variables.** OCaml *infers* every type. For `interleave` the toplevel
answers `'a -> 'a -> 'a -> 'a -> bool`: a function of four arguments of the same type `'a`,
returning a truth value. `'a` is a *type variable*. OCaml's comparisons `<`, `=`, `min` and `max`
work on values of any type, so nothing forces the arguments to be integers, and the function is
*polymorphic*: it works for every type `'a`. We will only use it with integers.

In [6]:
let interleave a b c d =
  let lo = min a b and hi = max a b in
  (lo < c && c < hi) <> (lo < d && d < hi)

let () = Printf.printf "%b %b\n%!" (interleave 0 2 1 3) (interleave 0 3 1 2)

val interleave : 'a -> 'a -> 'a -> 'a -> bool = <fun>


true false


`%b` prints a `bool`.

### No two arcs on one side cross

A river visits boundary points $P_0, P_1, \ldots$, and arc $k$ joins $P_k$ to $P_{k+1}$. Arcs
$j$ and $k$ are on the same side exactly when $j$ and $k$ have the same parity.

**Arrays, anonymous functions and higher-order functions.** `[| 0; 2; 1; 3 |]` is an *array*,
with constant-time access `p.(k)`. `fun x -> e` is an anonymous function. Functions are values:
`List.for_all f l` is true when `f x` is true for every `x` in the list `l`, and
`List.init n f` is the list `[f 0; f 1; …; f (n-1)]`. `Fun.id` is the identity function, so
`List.init n Fun.id` is `[0; 1; …; n-1]`.

In [7]:
let no_cross p l =
  List.for_all (fun k ->
      List.for_all (fun j ->
          j mod 2 <> k mod 2 || not (interleave p.(j) p.(j + 1) p.(k) p.(k + 1)))
        (List.init k Fun.id))
    (List.init l Fun.id)

let () = Printf.printf "%b %b\n%!" (no_cross [| 0; 1; 2; 3 |] 3) (no_cross [| 0; 2; 1; 3 |] 3)

val no_cross : 'a array -> int -> bool = <fun>


true false


The type of `no_cross` is `'a array -> int -> bool`: OCaml worked out from `p.(j)` that `p` is an
array, and from `j + 1` that its indices are integers; its entries can be of any type.

## 3. A river is a permutation

A meander is determined by the order in which the river crosses the bridges: its $i$-th
crossing is at bridge $\sigma(i)$, for a permutation $\sigma$ of $0, \ldots, n-1$. Treat the two
ends of the river, which go off to infinity, as two more boundary points east of every bridge:
the east end $n$ and the south end $n+1$. (Compactify each half-plane to a disk; the road plus
infinity is its boundary.) The path of the river is
$$n+1,\ \sigma(0),\ \ldots,\ \sigma(n-1),\ n,$$
with $n + 1$ arcs, the even ones below the road.

**Lists.** `[1; 2; 3]` is a list, `[]` the empty list, `x :: l` the list `l` with `x` in front,
and `l @ l'` concatenates. Lists are immutable and built from the front. `List.length l` is
the length of `l`, and `Array.of_list l` copies it into an array.

In [8]:
let path sigma =
  let n = List.length sigma in
  Array.of_list ((n + 1) :: sigma @ [ n ])

let is_meander sigma = no_cross (path sigma) (List.length sigma + 1)

val path : int list -> int array = <fun>


val is_meander : int list -> bool = <fun>


**Pattern matching and recursion.** `function | pattern -> e | …` is a function that looks at
the shape of its argument: `[]` matches the empty list, and `x :: rest` matches a non-empty one
and names its head and tail. A function that calls itself is declared with `let rec`.
`List.filter p l` keeps the entries satisfying `p`, `List.map f l` applies `f` to every entry,
and `List.concat_map f l` applies `f`, which returns lists, and concatenates the results.

All orderings of a list: pick each element `x` in turn to go first, and follow it with every
ordering of the others.

In [9]:
let rec permutations = function
  | [] -> [ [] ]
  | xs ->
      List.concat_map
        (fun x ->
          List.map (fun rest -> x :: rest) (permutations (List.filter (fun y -> y <> x) xs)))
        xs

val permutations : 'a list -> 'a list list = <fun>


**The pipeline operator.** `x |> f` is `f x`, so a chain of steps reads left to right.
`open_meander_count n` is the definition, word for word: the number of permutations that
describe a meander. It is the specification that everything faster must agree with.

In [10]:
let open_meander_count n =
  permutations (List.init n Fun.id) |> List.filter is_meander |> List.length

let small = List.init 9 open_meander_count

val open_meander_count : int -> int = <fun>


val small : int list = [1; 1; 1; 2; 3; 8; 14; 42; 81]


The toplevel prints the list as `[1; 1; 1; 2; 3; 8; 14; 42; 81]`.

## 4. Checking what we cannot prove

Lean proves that there is at least one meander and at most $n!$ for every $n$, and that its
fast counters agree with the definition. OCaml can check such facts for the values we try.

**`assert`, loops and sequencing.** `assert c` raises an exception, stopping the cell, if `c` is
false. `for i = a to b do body done` runs `body` for `i = a, …, b`; the body must have type
`unit`. `e1; e2` runs `e1` for its effect, then `e2`. A recursive function can be defined with
pattern matching on numbers too, using a guard: `| n when n > 0 -> …` matches only if the
condition holds, and `_` matches anything. `print_endline s` prints the string `s` and a new
line. `List.nth l i` is entry `i` of the list `l`.

In [11]:
let rec factorial = function n when n > 0 -> n * factorial (n - 1) | _ -> 1

let () =
  for n = 0 to 8 do
    let count = List.nth small n in
    assert (is_meander (List.init n Fun.id));
    assert (1 <= count && count <= factorial n)
  done;
  print_endline "checked n = 0, ..., 8"

val factorial : int -> int = <fun>


checked n = 0, ..., 8


The straight river `0, 1, …, n-1` is always a meander, which is how Lean proves there is at
least one.

These values are certified in Lean: by its kernel up to $n = 13$, by compiled code up to
$n = 24$ (`Arnold.openMeanderCount_values`, `Arnold.openMeanderCount_values_native`).
`List.filteri f l` keeps the entries whose position `i` and value pass `f i x`.

In [12]:
let lean_values =
  [ 1; 1; 1; 2; 3; 8; 14; 42; 81; 262; 538; 1828; 3926; 13820; 30694; 110954; 252939; 933458;
    2172830; 8152860; 19304190; 73424650; 176343390; 678390116; 1649008456 ]

let first k l = List.filteri (fun i _ -> i < k) l
let () = assert (small = first 9 lean_values)

val lean_values : int list =
  [1; 1; 1; 2; 3; 8; 14; 42; 81; 262; 538; 1828; 3926; 13820; 30694; 110954;
   252939; 933458; 2172830; 8152860; 19304190; 73424650; 176343390;
   678390116; 1649008456]


val first : int -> 'a list -> 'a list = <fun>


## 5. Closed meanders

A *closed* meander is a loop crossing the road $2n$ times. To count each loop once, start at the
westmost bridge $0$ and leave it upward. Arc $k$ joins $\sigma(k)$ to $\sigma(k+1)$, positions
taken modulo $2n$. `Array.init n f` is the array `[| f 0; …; f (n-1) |]`.

In [13]:
let is_closed_meander sigma =
  let m = List.length sigma in
  let s = Array.of_list sigma in
  let loop = Array.init (m + 1) (fun k -> s.(k mod m)) in
  s.(0) = 0 && no_cross loop m

let closed_meanders n =
  permutations (List.init (2 * n - 1) (fun i -> i + 1))
  |> List.map (fun rest -> 0 :: rest)
  |> List.filter is_closed_meander

let closed_counts = List.init 4 (fun i -> List.length (closed_meanders (i + 1)))

val is_closed_meander : int list -> bool = <fun>


val closed_meanders : int -> int list list = <fun>


val closed_counts : int list = [1; 2; 8; 42]


The counts $1, 2, 8, 42$ are the open counts for $1, 3, 5, 7$ crossings. Lean proves this for
every $n$: delete bridge $0$ from a closed meander and turn the picture through $180°$, and you
get an open meander with $2n - 1$ crossings. Conversely an open meander $\tau$ with $2n-1$
crossings becomes the closed meander $0, 2n - 1 - \tau(0), 2n - 1 - \tau(1), \ldots$ (Lean's
`closeUp`).

**Sorting and structural equality.** `List.sort compare l` sorts a list; `compare` is OCaml's
built-in ordering, which works on lists of integers too. Two sorted lists are equal with `=`
exactly when they hold the same elements, so this checks the bijection itself, not just the
counts.

In [14]:
let close_up tau =
  let n = List.length tau in
  0 :: List.map (fun t -> n - t) tau

let () =
  for n = 1 to 4 do
    let opens = permutations (List.init (2 * n - 1) Fun.id) |> List.filter is_meander in
    assert (List.sort compare (List.map close_up opens) = List.sort compare (closed_meanders n))
  done;
  print_endline "close_up is a bijection for n = 1, ..., 4"

val close_up : int list -> int list = <fun>


close_up is a bijection for n = 1, ..., 4


## 6. A search with a parity rule

A search builds the crossing order one bridge at a time and abandons a partial order as soon as
the newest arc crosses an earlier one on its side, or as soon as the parity rule fails.

**The parity rule.** Take an arc $j$ already drawn, on side $s$, with ends $a < b$. Every later
arc on side $s$ has both ends inside $(a, b)$ or both outside, since it may not cross arc $j$, so
the side-$s$ arc ends still to come inside $(a, b)$ pair up. They are the unvisited bridges in
$(a, b)$, the current point if the next arc is on side $s$, and the east end $n$ if the final
arc is on side $s$. An odd count means the river cannot be finished. Lean proves the rule sound
(`Arnold.parityOK_of_noCross`).

**Mutable state.** OCaml values are immutable unless declared otherwise. `ref v` is a mutable
cell holding `v`, `!r` reads it, `r := v` writes it, and `incr r`, `decr r` add or subtract one.
Arrays are mutable: `a.(i) <- v` writes an entry, and `Array.make n v` makes an array of `n`
copies of `v`. `begin … end` groups a sequence, like parentheses. `if c then a else b` is an
expression; without `else`, the branch must have type `unit`.

**Optional and labelled arguments.** `?(parity = true)` declares an optional argument with a
default; a caller writes `~parity:false` to set it. Functions defined inside `search` can see
its variables.

In [15]:
let search ?(parity = true) n =
  let order = Array.make n 0 and used = Array.make n false in
  let len = ref 0 and nodes = ref 0 in
  (* point k of the path: the south end, the bridges so far, then the east end *)
  let point k = if k = 0 then n + 1 else if k - 1 < !len then order.(k - 1) else n in
  let inside j x =
    let a = point j and b = point (j + 1) in
    min a b < x && x < max a b
  in
  let step_ok t =
    let c = point t and d = point (t + 1) in
    let rec ok j =
      j >= t
      || ((j mod 2 <> t mod 2 || not (interleave (point j) (point (j + 1)) c d)) && ok (j + 1))
    in
    ok 0
  in
  let parity_ok () =
    let t = !len in
    let rec ok j =
      j >= t
      ||
      let ends = ref 0 in
      for x = 0 to n - 1 do
        if (not used.(x)) && inside j x then incr ends
      done;
      if t mod 2 = j mod 2 && inside j (point t) then incr ends;
      if n mod 2 = j mod 2 && inside j n then incr ends;
      !ends mod 2 = 0 && ok (j + 1)
    in
    ok 0
  in
  let rec go () =
    incr nodes;
    if !len = n then if step_ok n then 1 else 0
    else begin
      let total = ref 0 in
      for x = 0 to n - 1 do
        if not used.(x) then begin
          order.(!len) <- x;
          used.(x) <- true;
          incr len;
          if step_ok (!len - 1) && ((not parity) || parity_ok ()) then total := !total + go ();
          decr len;
          used.(x) <- false
        end
      done;
      !total
    end
  in
  let count = go () in
  (count, !nodes)

val search : ?parity:bool -> int -> int * int = <fun>


**Libraries and timing.** Lines starting with `#` are *toplevel directives*, commands to the
toplevel itself; `;;` ends each one. `#use "topfind"` loads findlib, OCaml's library manager,
and `#require "unix"` then loads the `unix` library, whose `Unix.gettimeofday ()` reads the wall clock in seconds as a `float`.
`List.iter f l` calls `f` on every entry of `l`, for its effect. Below, `~parity` alone passes
the variable `parity` as the labelled argument of the same name, and `fst` takes the first part
of a pair.

In [16]:
#use "topfind";;
#require "unix";;

- : unit = ()


Findlib has been successfully loaded. Additional directives:


  #require "package";;      to load a package


  #list;;                   to list the available packages


  #camlp4o;;                to load camlp4 (standard syntax)


...

  Topfind.reset();;         to force that packages will be reloaded


  #thread;;                 to enable threads


- : unit = ()


/Users/alexy/.opam/5.5.1/lib/ocaml/unix: added to search path


/Users/alexy/.opam/5.5.1/lib/ocaml/unix/unix.cma: loaded


In [17]:
let () =
  List.iter
    (fun parity ->
      let t = Unix.gettimeofday () in
      let count, nodes = search ~parity 11 in
      Printf.printf "parity=%b: %d meanders, %d partial rivers, %.1f s\n%!" parity count nodes
        (Unix.gettimeofday () -. t))
    [ false; true ]

let () = assert (List.init 14 (fun n -> fst (search n)) = first 14 lean_values)

parity=false: 1828 meanders, 322080 partial rivers, 0.6 s


parity=true: 1828 meanders, 14471 partial rivers, 0.1 s


The parity rule cuts the partial rivers at eleven crossings from 322,080 to 14,471, as in the
paper.

## 7. The transfer matrix

The transfer matrix scans the road from west to east: the bridges $0, \ldots, m-1$, then the
east end $E = m$, which has one arc, on side $m \bmod 2$ (the south end, last, needs no step).
At each point and on each side, the point *opens* a new arc or *closes* an open one. Arcs on a
side never cross, so the open arcs on each side form a stack, and a point closes the top one.

**The state.** Cut the plane just east of the current point. The river west of the cut falls
into pieces, each joining two open arcs, or one open arc and the east end once it has been
passed. Read the open arcs along the cut from top to bottom. The pieces never cross, so they
pair the arcs like brackets. A state is a *word* of opening and closing brackets with at most
one `E` (an arc whose piece leads to the east end), and `h`, the number of arcs above the road.

**Variant types.** `type mark = Op | Cl | E` defines a new type with exactly three values, its
*constructors*. Pattern matching on a variant must cover every constructor; the compiler warns
otherwise.

**Records.** `type state = { word : mark list; h : int }` defines a record type with two named
fields. `{ word = w; h = 1 }` builds one, `s.h` reads a field, and the pattern `{ word; h }`
takes both fields apart, naming them after the fields. `show` prints a state as text:
`Printf.sprintf` formats into a string instead of printing, and `String.concat sep l` joins a
list of strings.

In [18]:
type mark = Op | Cl | E
type state = { word : mark list; h : int }

let show { word; h } =
  let char = function Op -> "(" | Cl -> ")" | E -> "E" in
  Printf.sprintf "%s/%d" (String.concat "" (List.map char word)) h

type mark = Op | Cl | E


type state = { word : mark list; h : int; }


val show : state -> string = <fun>


**Editing lists by position.** Lists have no positions to write into, so a change builds a new
list. Recursion walks to position `i` and rebuilds the front. `invalid_arg msg` raises an
exception for an impossible input, and `List.mapi f l` is `List.map` with positions.

In [19]:
let rec insert_at i x l =
  if i = 0 then x :: l
  else match l with y :: rest -> y :: insert_at (i - 1) x rest | [] -> invalid_arg "insert_at"

let rec remove_at i = function
  | [] -> invalid_arg "remove_at"
  | y :: rest -> if i = 0 then rest else y :: remove_at (i - 1) rest

let set_at i x l = List.mapi (fun j y -> if j = i then x else y) l

val insert_at : int -> 'a -> 'a list -> 'a list = <fun>


val remove_at : int -> 'a list -> 'a list = <fun>


val set_at : int -> 'a -> 'a list -> 'a list = <fun>


`partner w i` finds the bracket matched with the one at position `i`, scanning right from an
opening bracket or left from a closing one and skipping the `E`. The inner recursive function
carries the position and the depth as arguments instead of mutable variables.

In [20]:
let partner w i =
  let a = Array.of_list w in
  let step = if a.(i) = Op then 1 else -1 in
  let rec scan j depth =
    if a.(j) = a.(i) then scan (j + step) (depth + 1)
    else if a.(j) = E then scan (j + step) depth
    else if depth = 0 then j
    else scan (j + step) (depth - 1)
  in
  scan (i + step) 0

val partner : mark list -> int -> int = <fun>


**Callbacks, and matching on tuples.** `successors m x s emit` calls the function `emit` on
every state that point `x` can lead to, instead of building a list of them. `match a, b with`
matches two values at once, and `_` in a pattern matches anything.

* At a bridge, opening both sides puts a new pair at the cut, joined at the bridge.
* Opening above and closing below leaves the word alone: the closed arc's piece continues into
  the new arc. Only `h` changes; likewise the other way round.
* Closing both tops joins their pieces, unless the tops are partners, which would close a loop.
  Their partners become partners of each other, so a bracket may have to turn round.
* The east end opens or closes one arc on its side, and that piece now leads to `E`.

In [21]:
let successors m x { word; h } emit =
  let dn = List.length word - h in
  if x < m then begin
    emit { word = insert_at h Op (insert_at h Cl word); h = h + 1 };
    if dn > 0 then emit { word; h = h + 1 };
    if h > 0 then emit { word; h = h - 1 };
    if h > 0 && dn > 0 then begin
      let i = h - 1 and j = h in
      let a = List.nth word i and b = List.nth word j in
      if not (a = Op && b = Cl) then begin
        let w =
          match a, b with
          | E, _ -> set_at (partner word j) E word
          | _, E -> set_at (partner word i) E word
          | Op, Op -> set_at (partner word j) Op word
          | Cl, Cl -> set_at (partner word i) Cl word
          | _ -> word
        in
        emit { word = remove_at i (remove_at j w); h = h - 1 }
      end
    end
  end
  else begin
    let up = m mod 2 = 1 in
    emit { word = insert_at h E word; h = (if up then h + 1 else h) };
    let top = if up then h - 1 else h in
    if (if up then h > 0 else dn > 0) && List.nth word top <> E then
      emit { word = remove_at top (set_at (partner word top) E word);
             h = (if up then h - 1 else h) }
  end

val successors : int -> int -> state -> (state -> unit) -> unit = <fun>


A state is dropped as soon as a side has more open arcs than the remaining points can close
(Lean's `viable`).

In [22]:
let cap m k up = if k <= m then m - k + (if (m mod 2 = 1) = up then 1 else 0) else 0
let viable m k s = s.h <= cap m k true && List.length s.word - s.h <= cap m k false + 1

val cap : int -> int -> bool -> int = <fun>


val viable : int -> int -> state -> bool = <fun>


**Modules and functors.** A *module* groups definitions: `struct … end` writes one, and
`module Key = struct … end` names it. A *functor* is a function from modules to modules.
`Hashtbl.Make (Key)` builds a hash table specialized to keys of `Key.t`, using `Key.equal` and
`Key.hash`. The default hash looks only at the first few cells of a list, so long words would
collide; `Hashtbl.hash_param 100 100` looks much further. A *layer* maps each state to the
number of ways to reach it, and equal states are merged by adding their counts.

**`option`.** A value of type `'a option` is `None` or `Some v`. `States.find_opt t k` returns
`Some count` if `k` is in the table, and `Option.value o ~default:0` takes the value or `0`.
`States.replace t k v` sets an entry, `States.iter f t` calls `f k v` on every entry,
`States.create n` makes an empty table (`n` is a size hint), and `States.length t` counts its
entries. `layers m` collects every layer in a list, newest first, and `List.rev` reverses it. A
river is accepted when, after the east end, one arc is open, below the road, leading to `E`: the
state `E/0`.

In [23]:
module Key = struct
  type t = state
  let equal = ( = )
  let hash s = Hashtbl.hash_param 100 100 s
end

module States = Hashtbl.Make (Key)

let add tbl s c = States.replace tbl s (c + Option.value (States.find_opt tbl s) ~default:0)

let layers m =
  let layer = ref (States.create 1) in
  States.replace !layer { word = []; h = 0 } 1;
  let all = ref [ !layer ] in
  for x = 0 to m do
    let next = States.create (2 * States.length !layer) in
    States.iter
      (fun s c -> successors m x s (fun s' -> if viable m (x + 1) s' then add next s' c))
      !layer;
    layer := next;
    all := next :: !all
  done;
  List.rev !all

let final = { word = [ E ]; h = 0 }

let tm_count m =
  let last = List.nth (layers m) (m + 1) in
  Option.value (States.find_opt last final) ~default:0

module Key :
  sig type t = state val equal : 'a -> 'a -> bool val hash : 'a -> int end


module States :
  sig
    type key = Key.t
    type 'a t = 'a Hashtbl.Make(Key).t
    val create : int -> 'a t
    val clear : 'a t -> unit
    val reset : 'a t -> unit
    val copy : 'a t -> 'a t
    val add : 'a t -> key -> 'a -> unit
    val remove : 'a t -> key -> unit
    val find_and_remove : 'a t -> key -> 'a option
    val find : 'a t -> key -> 'a
    val find_opt : 'a t -> key -> 'a option
    val find_all : 'a t -> key -> 'a list
    val replace : 'a t -> key -> 'a -> unit
    val find_and_replace : 'a t -> key -> 'a -> 'a option
    val mem : 'a t -> key -> bool
    val iter : (key -> 'a -> unit) -> 'a t -> unit
    val filter_map_inplace : (key -> 'a -> 'a option) -> 'a t -> unit
    val fold : (key -> 'a -> 'acc -> 'acc) -> 'a t -> 'acc -> 'acc
    val length : 'a t -> int
    val stats : 'a t -> Hashtbl.statistics
    val to_seq : 'a t -> (key * 'a) Seq.t
    val to_seq_keys : 'a t -> key Seq.t
    val to_seq_values : 'a t -> 'a Seq.t
    val add_seq : 'a t -> (key * 'a) 

val add : int States.t -> States.key -> int -> unit = <fun>


val layers : int -> int States.t list = <fun>


val final : state = {word = [E]; h = 0}


val tm_count : int -> int = <fun>


The layers for three crossings:

`States.fold f t init` combines the entries of a table into one value, here a list of strings,
and `List.iteri` is `List.iter` that also passes each entry's position.

In [24]:
let () =
  List.iteri
    (fun k layer ->
      let entries = States.fold (fun s c acc -> Printf.sprintf "%s -> %d" (show s) c :: acc) layer [] in
      Printf.printf "%d: %s\n%!" k (String.concat ", " entries))
    (layers 3)

0: /0 -> 1


1: ()/1 -> 1


2: ()/0 -> 1, ()/2 -> 1, (())/2 -> 1


3: ()/1 -> 2


4: E/0 -> 2


In [25]:
let () =
  let t = Unix.gettimeofday () in
  let values = List.init 27 tm_count in
  Printf.printf "%.1f s; n = 26: %d\n%!" (Unix.gettimeofday () -. t) (List.nth values 26);
  assert (first 25 values = lean_values)

2.4 s; n = 26: 15730575554


The values agree with Lean's up to $n = 24$. OCaml's 63-bit integers hold the counts up to
$n = 42$; from $n = 43$ on, the compiled programs keep counts modulo two large numbers and
recover them by the Chinese remainder theorem.

## 8. All cores at once

OCaml 5 runs code in parallel on several *domains*, each an operating-system thread with its own
share of the runtime, all in one address space. The repository's Rust and OxCaml programs, and
the function below, share one algorithm:

* A layer is split into *shards* by a hash of the state, each shard its own table with its own
  lock.
* Workers take source shards one at a time from a shared counter. For every state they make the
  successors and send each to the shard its hash selects.
* A worker collects successors per target shard in a small buffer, and adds a full buffer to its
  shard under that shard's lock. Locks are taken rarely, and two workers block each other only
  when they flush to the same shard at the same moment.

**Domains, atomics and mutexes.** `Domain.spawn f` runs `f ()` on a new domain and returns a
handle; `Domain.join d` waits for it and returns its result.
`Domain.recommended_domain_count ()` is the number of cores. Plain mutable values shared between
domains are a *data race* if two domains touch them at once without synchronization. Two tools
prevent that. An `Atomic.t`, made by `Atomic.make v`, is a cell whose operations are indivisible:
`Atomic.fetch_and_add a 1` adds one and returns the old value, so every worker gets a different
shard number. A `Mutex.t`, made by `Mutex.create ()`, admits one domain at a time: `Mutex.protect m f` locks
`m`, runs `f`, and unlocks, even if `f` raises.

The shard comes from high bits of the hash, because the table inside a shard uses the low bits:
taking both from the same bits would crowd each shard's states into a few buckets. `lsr` shifts
right, and `land` is bitwise *and*. `Array.iteri` is `List.iteri` for arrays.

In [26]:
let shards = 64
let batch = 64
let shard_of s = (Key.hash s lsr 24) land (shards - 1)

let tm_count_parallel ?(domains = Domain.recommended_domain_count ()) m =
  let fresh () = Array.init shards (fun _ -> States.create 16) in
  let locks = Array.init shards (fun _ -> Mutex.create ()) in
  let layer = ref (fresh ()) in
  let start = { word = []; h = 0 } in
  States.replace !layer.(shard_of start) start 1;
  for x = 0 to m do
    let src = !layer and next = fresh () in
    let claim = Atomic.make 0 in
    let work () =
      let bufs = Array.make shards [] and sizes = Array.make shards 0 in
      let flush d =
        Mutex.protect locks.(d) (fun () -> List.iter (fun (s, c) -> add next.(d) s c) bufs.(d));
        bufs.(d) <- [];
        sizes.(d) <- 0
      in
      let rec loop () =
        let i = Atomic.fetch_and_add claim 1 in
        if i < shards then begin
          States.iter
            (fun s c ->
              successors m x s (fun s' ->
                  if viable m (x + 1) s' then begin
                    let d = shard_of s' in
                    bufs.(d) <- (s', c) :: bufs.(d);
                    sizes.(d) <- sizes.(d) + 1;
                    if sizes.(d) = batch then flush d
                  end))
            src.(i);
          loop ()
        end
      in
      loop ();
      Array.iteri (fun d b -> if b <> [] then flush d) bufs
    in
    let helpers = List.init (domains - 1) (fun _ -> Domain.spawn work) in
    work ();
    List.iter Domain.join helpers;
    layer := next
  done;
  Option.value (States.find_opt !layer.(shard_of final) final) ~default:0

val shards : int = 64


val batch : int = 64


val shard_of : 'a -> int = <fun>


val tm_count_parallel : ?domains:int -> int -> int = <fun>


The calling domain works too, so `domains - 1` helpers are spawned. Each worker's buffers are
its own, so only `claim`, the locks and the target tables are shared, and the tables are touched
only under their locks.

In [27]:
let () =
  List.iter
    (fun m ->
      let t0 = Unix.gettimeofday () in
      let a = tm_count m in
      let t1 = Unix.gettimeofday () in
      let b = tm_count_parallel m in
      let t2 = Unix.gettimeofday () in
      assert (a = b);
      Printf.printf "n = %d: %d   one domain %.1f s, %d domains %.1f s\n%!" m a (t1 -. t0)
        (Domain.recommended_domain_count ()) (t2 -. t1))
    [ 24; 28 ]

n = 24: 1649008456   one domain 0.3 s, 10 domains 0.1 s


n = 28: 152663683494   one domain 2.5 s, 10 domains 0.7 s


The notebook runs OCaml as bytecode in the toplevel, and its states are lists of variants, so
much of the time goes to allocation and garbage collection, which the domains share. The
compiled programs pack a state into one machine word instead.

## 9. OxCaml

[OxCaml](https://oxcaml.org/) is Jane Street's branch of the OCaml compiler. It adds
features for writing fast, safe parallel code. The repository's `oxcaml/` program,
`meanders_ox`, runs the transfer matrix of Section 7 on all cores with the sharded sweep of
Section 8. It packs each state into one integer, so it handles hundreds of millions of states
per layer. This section explains the OxCaml features it uses. Its code is shown here as text,
because this notebook's kernel runs standard OCaml 5. Build and run it with:

```text
cd oxcaml
opam exec --switch=5.2.0+ox -- dune build
./_build/default/bin/meanders_ox.exe 40 --from 36
```

**The state in one word.** The word of Section 7 needs only one bit per bracket (`1` for `(`),
the position of the `E` and `h`. Under a leading `1` that marks where the word ends, it fits in
one 63-bit OCaml integer: `h lsl 57 lor epos lsl 51 lor brackets`. Every transition becomes a
few shifts and masks on that integer, and a state costs no allocation at all.

**Modes.** OxCaml attaches *modes* to values, checked by the compiler alongside types, and
written `@ mode`.

* `local` values live on the stack and must not escape the function that made them. The
  default, `global`, lives on the garbage-collected heap. The hot loop allocates only local
  values, so it never triggers a collection.
* `portable` functions capture nothing that would be unsafe on another core, and may run
  there. `contended` values may be shared with other cores, so their mutable parts cannot be
  read or written; `uncontended` values belong to the current task. Together these modes rule
  out data races at compile time: two cores can never touch the same mutable data without
  synchronization.

**Capsules.** A capsule wraps mutable data that several cores share, with a mutex as its only
key. Each shard of a layer is a `Capsule.Sync.With_mutex.t`. A worker adds a batch to a shard
by locking it. The callback must be `portable`, so it can see the shard's table, but only
what it is handed and nothing it captured:

```oxcaml
let insert_batch (par @ local) (shard : shard) (ks : raw iarray @ local)
    (cs : raw iarray @ local) n =
  Capsule.Sync.With_mutex.with_lock (Parallel.sync par) shard ~f:(fun _ t ->
    T.add_batch t ks cs n)
```

The batch is passed as two immutable arrays (`iarray`) on the stack. The first version passed
the worker's mutable buffer instead, and the compiler rejected it: inside a `portable`
function that buffer is `contended`, so it cannot be read.

**The `Parallel` scheduler.** Instead of spawning domains by hand, the program runs inside
`Parallel_scheduler.with_parallel`, a work-stealing scheduler. A parallel loop gives each
worker its own slice of one scratch array. The slices are disjoint, and the type system knows
it, so each worker may write to its own:

```oxcaml
(S.fori [@kind float64]) par ~pivots ((S.slice [@kind float64]) scratch)
  ~f:(fun par _ buf -> worker par buf ~m ~x ~layer:layer_x ~next ~claim)
```

Workers claim source shards from a `Portable.Atomic` counter, exactly as the domains of
Section 8 do. A worker takes a claimed shard out of its capsule with `With_mutex.destroy`,
which hands it sole ownership, so it reads the shard without a lock and frees it.

**Unboxed types.** OCaml normally boxes values that are not small integers, which costs
allocation and pointer chasing. OxCaml adds *unboxed* types: `float#` and `int64#` are raw
machine numbers, `#(a * b * c * d)` is a tuple returned in registers, and arrays of them are
flat memory that the garbage collector never scans. `Word.successors` returns the up to four
successors of a state as an unboxed tuple, with `-1` for "none":

```oxcaml
let #(s0, s1, s2, s3) = Word.successors m x (k1 - 1) in
push par buf next s0 c;
push par buf next s1 c;
push par buf next s2 c;
push par buf next s3 c
```

**A store barrier, avoided safely.** On ARM processors, OCaml 5 places a memory barrier
before every store of a word into mutable memory. That keeps the language's guarantees for
programs that race, at about 10 ns per store. Stores of unboxed floats get no barrier. The
modes already prove the program race-free, so the tables keep each integer as its raw 64 bits
in a `float#` slot. One `fmov` instruction moves the bits each way, and no floating-point
arithmetic ever touches them. That single change made the single-thread sweep about 30% faster.

**How it compares.** On the development laptop (10 cores) the OxCaml program matches the
Rust program's wall time for $n = 38$ to $42$ (3.3 s against 3.6 s at $n = 40$), using a
little less CPU and somewhat more memory. Both reproduce the published counts, and both run
the machine whose correctness the Lean library proves. The repository's README has the larger
runs.

## 10. Index of OCaml features

Each feature is explained in the section listed, before its first use.

| Feature | Section |
|---|---|
| phrases, the toplevel's `- : type = value`, `(* comments *)` | 1 |
| `int`, `max_int`, `/`, `mod`, `float` and `+.`, `*.`, `/.` | 1 |
| tuples and their types `a * b` | 1 |
| `let`, `let … in`, `Printf.printf`, `%d`, `%s`, `%!`, `^` | 1 |
| destructuring `let q, r = …`, `let () = …`, `unit` | 1 |
| `bool`, `=`, `<>`, `&&`, `\|\|`, `not` | 1 |
| functions `let f a b = …`, `let … and …`, `<>` as exclusive or, `%b` | 2 |
| type inference, type variables `'a`, polymorphism | 2 |
| arrays `[\| … \|]`, `a.(i)`, `fun x -> e`, `List.for_all`, `List.init`, `Fun.id` | 2 |
| lists `[]`, `::`, `@`, `List.length`, `Array.of_list` | 3 |
| `function`, pattern matching, `let rec` | 3 |
| `List.filter`, `List.map`, `List.concat_map` | 3 |
| the pipeline operator `\|>` | 3 |
| `assert`, `for … do … done`, `;`, guards `when`, `_`, `print_endline`, `List.nth` | 4 |
| `List.filteri` | 4 |
| `Array.init`, `List.sort compare`, structural equality | 5 |
| `ref`, `!`, `:=`, `incr`, `decr`, `Array.make`, `a.(i) <- v`, `begin … end`, `if` | 6 |
| optional `?(x = v)` and labelled `~x` arguments | 6 |
| toplevel directives, `#use "topfind"`, `#require`, `Unix.gettimeofday`, `List.iter`, `fst` | 6 |
| variant types, constructors | 7 |
| records, field access, record patterns | 7 |
| `Printf.sprintf`, `String.concat`, `invalid_arg`, `List.mapi` | 7 |
| callbacks, matching on tuples | 7 |
| modules, `struct … end`, functors, `Hashtbl.Make`, `Hashtbl.hash_param` | 7 |
| `option`, `None`, `Some`, `Option.value`, `find_opt`, `replace`, `iter`, `create`, `length`, `List.rev` | 7 |
| `States.fold`, `List.iteri` | 7 |
| domains, `Domain.spawn`, `Domain.join`, `Domain.recommended_domain_count` | 8 |
| data races, `Atomic.make`, `Atomic.fetch_and_add`, `Mutex.create`, `Mutex.protect` | 8 |
| `lsr`, `land`, `Array.iteri` | 8 |
| OxCaml modes: `local`, `portable`, `contended`, `uncontended` | 9 |
| capsules, `With_mutex.with_lock`, `With_mutex.destroy` | 9 |
| the `Parallel` scheduler, parallel slices, `Portable.Atomic` | 9 |
| unboxed types `float#`, `int64#`, unboxed tuples `#( … )` | 9 |